# Section 0: Imports & Setup


In [2]:
import pandas as pd
import numpy as np
import torch
import os
import random
from torch import nn
from torch.utils.data import Dataset, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    BertTokenizer, BertModel, BertForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
import nltk
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
from nltk.corpus import wordnet
from google.colab import drive
drive.mount('/content/drive')

try:
    import torch_xla as torch_xla_pkg
    import torch_xla.core.xla_model as xm
    if not hasattr(torch, "xla"):
        torch.xla = torch_xla_pkg
    _TORCH_XLA_AVAILABLE = True
except Exception:
    xm = None
    _TORCH_XLA_AVAILABLE = False


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
#seed
SEED = random.randint(0, 4294967295)
print(f"Random seed: {SEED}")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


Random seed: 4051946914


# Section 1: WELFake 15k Subset Creation


In [4]:
WELFAKE_PATH = "/content/drive/MyDrive/datasets/WELFake_processed.csv"
WELFAKE_SUBSET_PATH = "/content/drive/MyDrive/datasets/WELFake_15k_subset.csv"
WELFAKE_PERTURBED_PATH = "/content/drive/MyDrive/datasets/WELFake_15k_subset_perturbed.csv"

# Subset generation
# df_welfake = pd.read_csv(WELFAKE_PATH).dropna()

# df_subset, _ = train_test_split(
#     df_welfake,
#     train_size=15000,
#     random_state=SEED,
#     stratify=df_welfake['label']
# )

# print(f"Subset size: {len(df_subset)}")
# print(df_subset['label'].value_counts())

# df_subset.to_csv(WELFAKE_SUBSET_PATH, index=False)
# print(f"Saved to: {WELFAKE_SUBSET_PATH}")


# Section 2: Text Perturbation Module


In [5]:
import subprocess
import sys

# Install nlpaug automatically if it's missing (Colab compatible)
try:
    import nlpaug.augmenter.word as naw
except ImportError:
    print("Installing nlpaug...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nlpaug"])
    import nlpaug.augmenter.word as naw

class TextPerturber:
    def __init__(self, delete_prob=0.1, swap_prob=0.1, substitute_prob=0.20):
        self.delete_prob = delete_prob
        self.swap_prob = swap_prob
        self.substitute_prob = substitute_prob

        # Apply synonym replacement first, then swapping, then deletion.
        self.aug_sub = naw.SynonymAug(aug_src='wordnet', aug_p=self.substitute_prob)
        self.aug_swap = naw.RandomWordAug(action="swap", aug_p=self.swap_prob)
        self.aug_del = naw.RandomWordAug(action="delete", aug_p=self.delete_prob)

    def _first_value(self, augmented_text):
        if isinstance(augmented_text, list):
            return augmented_text[0]
        return augmented_text

    def perturb(self, text: str) -> str:
        # 1. Replace words with WordNet synonyms.
        augmented_text = self._first_value(self.aug_sub.augment(text))

        # 2. Swap a small fraction of words.
        augmented_text = self._first_value(self.aug_swap.augment(augmented_text))

        # 3. Delete a small fraction of words.
        augmented_text = self._first_value(self.aug_del.augment(augmented_text))

        return augmented_text

Installing nlpaug...


In [6]:
# Section 2.1: Precompute Perturbed Texts (Run once)
PRECOMPUTE_PERTURBATIONS = False

if PRECOMPUTE_PERTURBATIONS:
    if os.path.exists(WELFAKE_PERTURBED_PATH):
        print(f"Perturbed file already exists: {WELFAKE_PERTURBED_PATH}")
    else:
        df = pd.read_csv(WELFAKE_SUBSET_PATH).dropna()
        perturber = TextPerturber()
        df["combined_text_perturbed"] = [
            perturber.perturb(t) for t in df["combined_text"].tolist()
        ]
        df.to_csv(WELFAKE_PERTURBED_PATH, index=False)
        print(f"Saved perturbed dataset: {WELFAKE_PERTURBED_PATH}")


# Section 3: Dataset Classes


In [7]:
class FakeNewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx], dtype=torch.long) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
    def __len__(self):
        return len(self.labels)

class AdversarialFakeNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, perturber=None, perturbed_texts=None, max_length=128):
        if perturbed_texts is None:
            if perturber is None:
                raise ValueError("perturber is required when perturbed_texts is not provided")
            perturbed_texts = [perturber.perturb(t) for t in texts]

        self.orig_enc = tokenizer(texts, truncation=True,
                                  padding='max_length', max_length=max_length)
        self.pert_enc = tokenizer(perturbed_texts, truncation=True,
                                  padding='max_length', max_length=max_length)
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx], dtype=torch.long) for k, v in self.orig_enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        item['input_ids_pert'] = torch.tensor(self.pert_enc['input_ids'][idx], dtype=torch.long)
        item['attention_mask_pert'] = torch.tensor(self.pert_enc['attention_mask'][idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)


# Section 4: Model Architecture & Loss


In [8]:
class AdversarialBERT(nn.Module):
    def __init__(self, num_labels=2, dropout=0.1):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(768, num_labels)

    def gradient_checkpointing_enable(self, **kwargs):
        self.bert.gradient_checkpointing_enable(**kwargs)

    def gradient_checkpointing_disable(self):
        self.bert.gradient_checkpointing_disable()

    def forward(self, input_ids, attention_mask, token_type_ids=None, **kwargs):
        if token_type_ids is not None:
            out = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        else:
            out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_emb = out.last_hidden_state[:, 0, :]   # [CLS] token, shape [B, 768]
        logits = self.classifier(self.dropout(cls_emb))
        return logits, cls_emb

def adversarial_loss(logits_orig, logits_pert, labels, cls_orig, cls_pert, lambda_adv=0.5):
    ce = nn.CrossEntropyLoss()
    ce_loss = 0.5 * (ce(logits_orig, labels) + ce(logits_pert, labels))
    target = torch.ones(cls_orig.size(0), device=cls_orig.device)
    adv_loss = nn.CosineEmbeddingLoss()(cls_orig, cls_pert, target)
    return ce_loss + (lambda_adv * adv_loss)

# Section 5: Custom Trainer


In [9]:
from transformers import TrainerCallback

class LambdaSchedulerCallback(TrainerCallback):
    def __init__(self, max_lambda=0.5, warmup_ratio=0.1):
        self.max_lambda = max_lambda
        self.warmup_ratio = warmup_ratio
        self.trainer = None

    def on_step_begin(self, args, state, control, **kwargs):
        trainer = kwargs.get('trainer') or self.trainer
        if trainer is None:
            return

        if not hasattr(trainer, 'lambda_adv'):
            trainer.lambda_adv = 0.0

        total_steps = state.max_steps
        current_step = state.global_step

        if total_steps <= 0:
            return

        warmup_steps = total_steps * self.warmup_ratio
        if warmup_steps <= 0:
            trainer.lambda_adv = self.max_lambda
            return

        if current_step < warmup_steps:
            trainer.lambda_adv = self.max_lambda * (current_step / warmup_steps)
        else:
            trainer.lambda_adv = self.max_lambda

class AdversarialTrainer(Trainer):
    def __init__(self, *args, lambda_adv=0.5, **kwargs):
        super().__init__(*args, **kwargs)
        self.lambda_adv = lambda_adv

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels           = inputs.get('labels')
        input_ids_pert   = inputs.get('input_ids_pert')
        attn_mask_pert   = inputs.get('attention_mask_pert')

        # Concatenate original and perturbed inputs along the batch dimension
        combined_input_ids = torch.cat([inputs['input_ids'], input_ids_pert], dim=0)
        combined_attention_mask = torch.cat([inputs['attention_mask'], attn_mask_pert], dim=0)
        
        combined_token_type_ids = None
        if 'token_type_ids' in inputs:
            # Duplicate the token_type_ids for the perturbed half
            combined_token_type_ids = torch.cat([inputs['token_type_ids'], inputs['token_type_ids']], dim=0)

        # 1 Single Forward Pass
        combined_logits, combined_cls = model(
            input_ids=combined_input_ids, 
            attention_mask=combined_attention_mask, 
            token_type_ids=combined_token_type_ids
        )

        # Split outputs back into original and perturbed sets physically
        batch_size = labels.size(0)
        logits_orig, logits_pert = combined_logits[:batch_size], combined_logits[batch_size:]
        cls_orig, cls_pert = combined_cls[:batch_size], combined_cls[batch_size:]

        loss = adversarial_loss(logits_orig, logits_pert, labels, cls_orig, cls_pert, self.lambda_adv)
        return (loss, (loss, logits_orig)) if return_outputs else loss

In [10]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary',
    )
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

def load_device():
    if _TORCH_XLA_AVAILABLE and xm is not None:
        try:
            device = xm.xla_device()
            print(f"✓ Using TPU: {device}")
            return device, True
        except Exception:
            pass
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"✓ Using CUDA: {torch.cuda.get_device_name(0)}")
    else:
        device = torch.device("cpu")
        print("⚠ Using CPU (Training will be slow!)")
    return device, False


# Section 6: Smoke Test (Sanity Check)


In [11]:
# Quick test to ensure adversarial training passes gradients correctly
# Set to True to run
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    print("Running Smoke Test...")
    device, use_tpu = load_device()
    df = pd.read_csv(WELFAKE_SUBSET_PATH).dropna().head(200)
    
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    perturber = TextPerturber()
    
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        df['combined_text'].tolist(), df['label'].tolist(),
        test_size=0.20, random_state=SEED, stratify=df['label']
    )
    
    train_dataset = AdversarialFakeNewsDataset(train_texts, train_labels, tokenizer, perturber)
    val_dataset = AdversarialFakeNewsDataset(val_texts, val_labels, tokenizer, perturber)
    
    model = AdversarialBERT()
    if not use_tpu:
        model.to(device)
        
    training_kwargs = {
        "output_dir": "./results_smoke",
        "num_train_epochs": 1,
        "per_device_train_batch_size": 16,
        "per_device_eval_batch_size": 16,
        "save_strategy": "no",
        "report_to": "none",
        "optim": "adamw_torch",
        "logging_steps": 5,
        "remove_unused_columns": False,
        "label_names": ["labels"],
        "gradient_checkpointing": False if use_tpu else True,
    }
    if "evaluation_strategy" in TrainingArguments.__init__.__code__.co_varnames:
        training_kwargs["evaluation_strategy"] = "no"
    else:
        training_kwargs["eval_strategy"] = "no"

    training_args = TrainingArguments(**training_kwargs)
    
    trainer = AdversarialTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        lambda_adv=0.5
    )
    
    trainer.train()
    print("Smoke test passed!")


# Experiment Setup


In [12]:
def run_experiment(exp_name, dataset_class, is_adversarial=False, perturber_config=None):
    device, use_tpu = load_device()
    df = pd.read_csv(WELFAKE_SUBSET_PATH).dropna()
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

    indices = list(range(len(df)))
    train_idx, val_idx = train_test_split(
        indices, test_size=0.20, random_state=SEED, stratify=df['label']
    )
    train_texts = [df['combined_text'].iloc[i] for i in train_idx]
    val_texts = [df['combined_text'].iloc[i] for i in val_idx]
    train_labels = [df['label'].iloc[i] for i in train_idx]
    val_labels = [df['label'].iloc[i] for i in val_idx]
    print(f"Train: {len(train_texts)} | Val: {len(val_texts)}")
    
    if is_adversarial:
        # Optionally load precomputed perturbed texts
        pert_texts = None
        if os.path.exists(WELFAKE_PERTURBED_PATH):
            pert_df = pd.read_csv(WELFAKE_PERTURBED_PATH).dropna()
            if ("combined_text_perturbed" in pert_df.columns) and (len(pert_df) == len(df)):
                pert_texts = pert_df["combined_text_perturbed"].tolist()
        train_pert_texts = [pert_texts[i] for i in train_idx] if pert_texts else None
        val_pert_texts = [pert_texts[i] for i in val_idx] if pert_texts else None

        # Build perturber according to provided config (or default)
        if perturber_config is None:
            perturber = None if pert_texts else TextPerturber()
        else:
            perturber = None if pert_texts else TextPerturber(**perturber_config)

        train_dataset = dataset_class(
            train_texts, train_labels, tokenizer, perturber,
            perturbed_texts=train_pert_texts
        )
        val_dataset = dataset_class(
            val_texts, val_labels, tokenizer, perturber,
            perturbed_texts=val_pert_texts
        )
        model = AdversarialBERT()
        TrainerClass = AdversarialTrainer
    else:
        train_encodings = tokenizer(train_texts, truncation=True, padding='max_length', max_length=128)
        val_encodings = tokenizer(val_texts, truncation=True, padding='max_length', max_length=128)
        train_dataset = dataset_class(train_encodings, train_labels)
        val_dataset = dataset_class(val_encodings, val_labels)
        model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
        TrainerClass = Trainer

    if not use_tpu:
        model.to(device)

    training_kwargs = {
        "output_dir": f"./results_{exp_name}",
        "num_train_epochs": 3,
        "per_device_train_batch_size": 32 if is_adversarial else 64,
        "per_device_eval_batch_size": 32 if is_adversarial else 64,
        "gradient_accumulation_steps": 2 if is_adversarial else 2,
        "save_strategy": "steps",
        "save_steps": 128,
        "save_total_limit": 2,
        "bf16": use_tpu,
        "gradient_checkpointing": (is_adversarial and not use_tpu),
        "report_to": "none",
        "optim": "adamw_torch",
        "logging_steps": 128,
        "metric_for_best_model": "f1",
        "load_best_model_at_end": True,
        "weight_decay": 0.01,
        "remove_unused_columns": False,
        "label_names": ["labels"],
    }
    if "evaluation_strategy" in TrainingArguments.__init__.__code__.co_varnames:
        training_kwargs["evaluation_strategy"] = "steps"
        training_kwargs["eval_steps"] = 128
    else:
        training_kwargs["eval_strategy"] = "steps"
        training_kwargs["eval_steps"] = 128

    training_args = TrainingArguments(**training_kwargs)

    if is_adversarial:
        trainer = TrainerClass(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            compute_metrics=compute_metrics,
            callbacks=[
                EarlyStoppingCallback(early_stopping_patience=3),
                LambdaSchedulerCallback(max_lambda=0.5, warmup_ratio=0.1)
            ],
            lambda_adv=0.0
        )
        for callback in trainer.callback_handler.callbacks:
            if isinstance(callback, LambdaSchedulerCallback):
                callback.trainer = trainer
    else:
        trainer = TrainerClass(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
        )

    print(f"\n--- Running {exp_name} ---")
    trainer.train()
    
    val_metrics = trainer.evaluate()
    print(f"Validation Metrics: {val_metrics}")
    
    return model, trainer, val_metrics

# Section 7: Experiment 0 - Baseline BERT


In [12]:
base_model, base_trainer, base_val_metrics = run_experiment('Baseline', FakeNewsDataset, is_adversarial=False)


/tmp/ipykernel_2694/2398473430.py:18: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()


✓ Using TPU: xla:0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Validation Metrics: {'eval_loss': 0.06834378838539124, 'eval_accuracy': 0.977, 'eval_f1': 0.9748633879781421, 'eval_precision': 0.9667630057803468, 'eval_recall': 0.9831006612784717, 'eval_runtime': 1.4927, 'eval_samples_per_second': 2015.114, 'eval_steps_per_second': 31.486, 'epoch': 3.0}


# Section 8: Experiment 1 - Adversarial BERT


In [13]:
adv_model, adv_trainer, adv_val_metrics = run_experiment('Adversarial', AdversarialFakeNewsDataset, is_adversarial=True)


/tmp/ipykernel_2694/2398473430.py:18: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()


✓ Using TPU: xla:0
Train: 12000 | Val: 3000


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Running Adversarial ---


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
128,0.404297,0.084926,0.973000,0.970811,0.952617,0.989713
256,0.210938,0.060608,0.977000,0.974660,0.974302,0.975018
384,0.150391,0.055392,0.979667,0.977631,0.975842,0.979427
512,0.143555,0.055060,0.980667,0.978739,0.976591,0.980896


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Validation Metrics: {'eval_loss': 0.05506020411849022, 'eval_accuracy': 0.9806666666666667, 'eval_f1': 0.9787390029325513, 'eval_precision': 0.9765910753474762, 'eval_recall': 0.9808963997060984, 'eval_runtime': 3.3138, 'eval_samples_per_second': 907.721, 'eval_steps_per_second': 28.366, 'epoch': 3.0}


# Section 9: Results Comparison


In [14]:
print("Baseline Results:")
print(base_val_metrics)

print("Adversarial Results:")
print(adv_val_metrics)



Baseline Results:
{'eval_loss': 0.06834378838539124, 'eval_accuracy': 0.977, 'eval_f1': 0.9748633879781421, 'eval_precision': 0.9667630057803468, 'eval_recall': 0.9831006612784717, 'eval_runtime': 1.4927, 'eval_samples_per_second': 2015.114, 'eval_steps_per_second': 31.486, 'epoch': 3.0}
Adversarial Results:
{'eval_loss': 0.05506020411849022, 'eval_accuracy': 0.9806666666666667, 'eval_f1': 0.9787390029325513, 'eval_precision': 0.9765910753474762, 'eval_recall': 0.9808963997060984, 'eval_runtime': 3.3138, 'eval_samples_per_second': 907.721, 'eval_steps_per_second': 28.366, 'epoch': 3.0}


# Section 10: Cross-Dataset Generalization


In [14]:
def cross_dataset_evaluation(model, tokenizer, is_adversarial, all_datasets_paths, compute_metrics_fn):
    from transformers.modeling_outputs import SequenceClassifierOutput
    device, use_tpu = load_device()
    
    print("\n" + "=" * 60)
    print("CROSS-DATASET GENERALIZATION")
    print("=" * 60)

    # Wrapper to make custom models compatible with standard Trainer evaluation
    class ModelWrapper(nn.Module):
        def __init__(self, inner_model):
            super().__init__()
            self.inner_model = inner_model
        def forward(self, input_ids, attention_mask, labels=None, **kwargs):
            # Handle different return types (AdversarialBERT returns tuple, BertForSequenceClassification returns ModelOutput)
            outputs = self.inner_model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs[0] if isinstance(outputs, (tuple, list)) else outputs.logits
            
            loss = None
            if labels is not None:
                loss = nn.CrossEntropyLoss()(logits, labels)
            
            return SequenceClassifierOutput(loss=loss, logits=logits)

    # Wrap the model
    eval_model = ModelWrapper(model)
    
    results = {}

    for name, path in all_datasets_paths.items():
        if name == "WELFake": # Skip training dataset
            continue

        print(f"\nTesting on unseen dataset: {name} (Full Dataset)...")
        df = pd.read_csv(path).dropna()

        test_texts = df['combined_text'].tolist()
        test_labels = df['label'].tolist()

        encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)
        dataset = FakeNewsDataset(encodings, test_labels)

        eval_trainer = Trainer(
            model=eval_model,
            compute_metrics=compute_metrics_fn,
            args=TrainingArguments(
                output_dir="./temp_eval",
                remove_unused_columns=False,
                label_names=["labels"],
                per_device_eval_batch_size=32,
                report_to="none"
            )
        )

        metrics = eval_trainer.evaluate(eval_dataset=dataset)
        results[name] = metrics
        
        acc = metrics.get('eval_accuracy', metrics.get('accuracy', 0))
        f1 = metrics.get('eval_f1', metrics.get('f1', 0))
        
        print(f"  -> {name} Accuracy: {acc:.4f}, F1: {f1:.4f}")

    return results

DATASETS = {
    "WELFake": "/content/drive/MyDrive/datasets/WELFake_processed.csv",
    "FakeNewsNet": "/content/drive/MyDrive/datasets/FakeNewsNet_processed.csv",
    "Fake_News_Detection": "/content/drive/MyDrive/datasets/Fake_News_Detection_processed.csv",
    "ISOT": "/content/drive/MyDrive/datasets/ISOT_processed.csv",
    "Fake_News_Classification": "/content/drive/MyDrive/datasets/Fake_News_Classification_processed.csv"
}

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')


In [16]:
adv_cross_results = cross_dataset_evaluation(adv_model, tokenizer, True, DATASETS, compute_metrics)


/tmp/ipykernel_2694/2398473430.py:18: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()


✓ Using TPU: xla:0

CROSS-DATASET GENERALIZATION

Testing on unseen dataset: FakeNewsNet (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> FakeNewsNet Accuracy: 0.7152, F1: 0.8307

Testing on unseen dataset: Fake_News_Detection (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Detection Accuracy: 0.1058, F1: 0.1837

Testing on unseen dataset: ISOT (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> ISOT Accuracy: 0.9945, F1: 0.9939

Testing on unseen dataset: Fake_News_Classification (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Classification Accuracy: 0.0232, F1: 0.0360


In [17]:
base_cross_results = cross_dataset_evaluation(
    model=base_model, 
    tokenizer=tokenizer, 
    is_adversarial=False, 
    all_datasets_paths=DATASETS, 
    compute_metrics_fn=compute_metrics
)

/tmp/ipykernel_2694/2398473430.py:18: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()


✓ Using TPU: xla:0

CROSS-DATASET GENERALIZATION

Testing on unseen dataset: FakeNewsNet (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> FakeNewsNet Accuracy: 0.6873, F1: 0.8091

Testing on unseen dataset: Fake_News_Detection (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Detection Accuracy: 0.1090, F1: 0.1885

Testing on unseen dataset: ISOT (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> ISOT Accuracy: 0.9942, F1: 0.9937

Testing on unseen dataset: Fake_News_Classification (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Classification Accuracy: 0.0234, F1: 0.0360


# Perturbation Optimisation

In [ ]:
# Experiment: 10% perturbation (delete+swap = 5% each)
pert_config = {"delete_prob": 0.05, "swap_prob": 0.05, "substitute_prob": 0.20}

exp_name = "Adversarial_10pct"

model_10, trainer_10, metrics_10 = run_experiment(exp_name, AdversarialFakeNewsDataset, is_adversarial=True, perturber_config=pert_config)
cross_10 = cross_dataset_evaluation(model_10, tokenizer, True, DATASETS, compute_metrics)

pd.DataFrame.from_dict(cross_10, orient="index").to_csv(f"./results_{exp_name}_cross.csv")
print("Cross-dataset results for", exp_name)
print(pd.DataFrame.from_dict(cross_10, orient="index"))

/tmp/ipykernel_2694/2398473430.py:18: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()


✓ Using TPU: xla:0
Train: 12000 | Val: 3000


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Running Adversarial_10pct ---


...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> ISOT Accuracy: 0.9933, F1: 0.9927

Testing on unseen dataset: Fake_News_Classification (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Classification Accuracy: 0.0243, F1: 0.0360
Cross-dataset results for Adversarial_10pct
                          eval_loss  eval_model_preparation_time  \
FakeNewsNet                0.954754                       0.0019   
Fake_News_Detection        4.457601                       0.0025   
ISOT                       0.019111                       0.0033   
Fake_News_Classification   5.858131                       0.0035   

                          eval_accuracy   eval_f1  eval_precision  \
FakeNewsNet                    0.705365  0.823468        0.752959   
Fake_News_Detection            0.093920  0.161960        0.164320   
ISOT                           0.993324  0.992661        0.999434   
Fake_News_Classification       0.024347  0.036034        0.038644   

                          eval_recall  eval_runtime  eval_samples_per_second  \
FakeNewsNet                  0.908546       81.4393                   67.093   
Fake_News_Detection          0.159668      155.496

: 

In [ ]:
# Experiment: 30% perturbation (delete+swap = 15% each)
pert_config = {"delete_prob": 0.15, "swap_prob": 0.15, "substitute_prob": 0.20}

exp_name = "Adversarial_30pct"

model_30, trainer_30, metrics_30 = run_experiment(exp_name, AdversarialFakeNewsDataset, is_adversarial=True, perturber_config=pert_config)
# cross_30 = cross_dataset_evaluation(model_30, tokenizer, True, DATASETS, compute_metrics)

# pd.DataFrame.from_dict(cross_30, orient="index").to_csv(f"./results_{exp_name}_cross.csv")
# print("Cross-dataset results for", exp_name)
# print(pd.DataFrame.from_dict(cross_30, orient="index"))

/tmp/ipykernel_3810/2398473430.py:18: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()


✓ Using TPU: xla:0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
128,0.447266,0.094618,0.966000,0.962909,0.953204,0.972814
256,0.210938,0.070704,0.973000,0.970384,0.965793,0.975018
384,0.155273,0.067513,0.974667,0.972141,0.970007,0.974284
512,0.140625,0.067218,0.975333,0.972914,0.969365,0.976488


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Validation Metrics: {'eval_loss': 0.06721756607294083, 'eval_accuracy': 0.9753333333333334, 'eval_f1': 0.972913616398243, 'eval_precision': 0.9693654266958425, 'eval_recall': 0.9764878765613519, 'eval_runtime': 3.0723, 'eval_samples_per_second': 979.065, 'eval_steps_per_second': 30.596, 'epoch': 3.0}


NameError: name 'cross_dataset_evaluation' is not defined

In [15]:
cross_30 = cross_dataset_evaluation(model_30, tokenizer, True, DATASETS, compute_metrics)

pd.DataFrame.from_dict(cross_30, orient="index").to_csv(f"./results_{exp_name}_cross.csv")
print("Cross-dataset results for", exp_name)
print(pd.DataFrame.from_dict(cross_30, orient="index"))

/tmp/ipykernel_3810/2398473430.py:18: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()


✓ Using TPU: xla:0

CROSS-DATASET GENERALIZATION

Testing on unseen dataset: FakeNewsNet (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> FakeNewsNet Accuracy: 0.7086, F1: 0.8265

Testing on unseen dataset: Fake_News_Detection (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Detection Accuracy: 0.0885, F1: 0.1541

Testing on unseen dataset: ISOT (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> ISOT Accuracy: 0.9942, F1: 0.9937

Testing on unseen dataset: Fake_News_Classification (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Classification Accuracy: 0.0235, F1: 0.0360
Cross-dataset results for Adversarial_30pct
                          eval_loss  eval_model_preparation_time  \
FakeNewsNet                0.985579                       0.0019   
Fake_News_Detection        4.419857                       0.0021   
ISOT                       0.016853                       0.0059   
Fake_News_Classification   5.879777                       0.0031   

                          eval_accuracy   eval_f1  eval_precision  \
FakeNewsNet                    0.708570  0.826549        0.751635   
Fake_News_Detection            0.088512  0.154105        0.156896   
ISOT                           0.994220  0.993651        0.999435   
Fake_News_Classification       0.023484  0.035957        0.038526   

                          eval_recall  eval_runtime  eval_samples_per_second  \
FakeNewsNet                  0.918049       68.4660                   79.806   
Fake_News_Detection          0.151411      132.212

In [16]:
# Experiment: 40% perturbation (delete+swap = 20% each)
pert_config = {"delete_prob": 0.20, "swap_prob": 0.20, "substitute_prob": 0.20}

exp_name = "Adversarial_40pct"

model_40, trainer_40, metrics_40 = run_experiment(exp_name, AdversarialFakeNewsDataset, is_adversarial=True, perturber_config=pert_config)
cross_40 = cross_dataset_evaluation(model_40, tokenizer, True, DATASETS, compute_metrics)

pd.DataFrame.from_dict(cross_40, orient="index").to_csv(f"./results_{exp_name}_cross.csv")
print("Cross-dataset results for", exp_name)
print(pd.DataFrame.from_dict(cross_40, orient="index"))

/tmp/ipykernel_3810/2398473430.py:18: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()


✓ Using TPU: xla:0
Train: 12000 | Val: 3000


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Running Adversarial_40pct ---


...

  -> Fake_News_Detection Accuracy: 0.1061, F1: 0.1838

Testing on unseen dataset: ISOT (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> ISOT Accuracy: 0.9941, F1: 0.9935

Testing on unseen dataset: Fake_News_Classification (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Classification Accuracy: 0.0235, F1: 0.0360
Cross-dataset results for Adversarial_40pct
                          eval_loss  eval_model_preparation_time  \
FakeNewsNet                0.929835                       0.0020   
Fake_News_Detection        4.471557                       0.0032   
ISOT                       0.017044                       0.0068   
Fake_News_Classification   5.860249                       0.0053   

                          eval_accuracy   eval_f1  eval_precision  \
FakeNewsNet                    0.706098  0.824273        0.752399   
Fake_News_Detection            0.106132  0.183764        0.184034   
ISOT                           0.994092  0.993510        0.999435   
Fake_News_Classification       0.023484  0.035957        0.038526   

                          eval_recall  eval_runtime  eval_samples_per_second  \
FakeNewsNet                  0.911330       77.1150                   70.855   
Fake_News_Detection          0.183495      147.370

In [ ]:
# Experiment: 50% perturbation (delete+swap = 25% each)
pert_config = {"delete_prob": 0.25, "swap_prob": 0.25, "substitute_prob": 0.20}

exp_name = "Adversarial_50pct"

model_50, trainer_50, metrics_50 = run_experiment(exp_name, AdversarialFakeNewsDataset, is_adversarial=True, perturber_config=pert_config)
cross_50 = cross_dataset_evaluation(model_50, tokenizer, True, DATASETS, compute_metrics)

pd.DataFrame.from_dict(cross_50, orient="index").to_csv(f"./results_{exp_name}_cross.csv")
print("Cross-dataset results for", exp_name)
print(pd.DataFrame.from_dict(cross_50, orient="index"))

/tmp/ipykernel_3810/2398473430.py:18: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()


✓ Using TPU: xla:0
Train: 12000 | Val: 3000


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Running Adversarial_50pct ---


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
128,0.406250,0.083305,0.973000,0.970253,0.969897,0.970610
256,0.186523,0.077042,0.976000,0.973684,0.968727,0.978692
384,0.144531,0.075496,0.975333,0.972814,0.972814,0.972814
512,0.135742,0.073890,0.975333,0.972774,0.974208,0.971345


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Validation Metrics: {'eval_loss': 0.07704195380210876, 'eval_accuracy': 0.976, 'eval_f1': 0.9736842105263158, 'eval_precision': 0.9687272727272728, 'eval_recall': 0.9786921381337252, 'eval_runtime': 3.5578, 'eval_samples_per_second': 845.455, 'eval_steps_per_second': 26.42, 'epoch': 3.0}
✓ Using TPU: xla:0

CROSS-DATASET GENERALIZATION

Testing on unseen dataset: FakeNewsNet (Full Dataset)...


/tmp/ipykernel_3810/2398473430.py:18: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
